# Convolutionnal Neural Network

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import pandas as pd
from tokenizers import Tokenizer, models, trainers
from tokenizers.pre_tokenizers import Whitespace

from torchvision import datasets, transforms, utils

1. Reading the csv file

In [5]:
dataset = pd.read_csv('rotten_tomatoes_movie_reviews.csv')
for index, row in dataset.head(10).iterrows():
    print(f"Review: {row['reviewText']}")
    print(f"Sentiment: {row['scoreSentiment']}")
    print()

print(f"Number of reviews in the dataset: {len(dataset)}")

Review: Timed to be just long enough for most youngsters' brief attention spans -- and it's packed with plenty of interesting activity, both on land and under the water.
Sentiment: POSITIVE

Review: It doesn't matter if a movie costs 300 million or only 300 dollars; good is good and bad is bad, and Bloodmask: The Possession of Nicole Lameroux is just plain bad.
Sentiment: NEGATIVE

Review: The choreography is so precise and lifelike at points one might wonder whether the movie was rotoscoped, but no live-action reference footage was used. The quality is due to the skill of the animators and Kodama's love for professional wrestling.
Sentiment: POSITIVE

Review: The film's out-of-touch attempts at humor may find them hunting for the reason the franchise was so popular in the first place.
Sentiment: NEGATIVE

Review: Its clumsy determination is endearing and sometimes wildly entertaining
Sentiment: POSITIVE

Review: With every new minute, there's another head-scratching choice that's boun

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


__Hyperparameters__

In [6]:
vocab_n = 10000
sequence_length = 64
batch_size = 8
num_workers = 0

learning_rate = 1e-3
num_epochs = 20

__Dataset defining__

In [9]:
class SentimentDataset(data.Dataset):
    def __init__(self, root_dir, train=True):
        super().__init__()
        dataSet = pd.read_csv(root_dir)

        tokenizer = Tokenizer(models.BPE())
        tokenizer.pre_tokenizer = Whitespace()
        tokenizer.enable_padding(max_length=sequence_length)
        tokenizer.enable_truncation(max_length=sequence_length)
        tokenizer_trainer = trainers.BpeTrainer(vocab_size=vocab_n)


        data = []
        for index, row in dataSet.iterrows():
            if not isinstance(row['reviewText'], str):
                continue
            if row['scoreSentiment'] == "POSITIVE":
                data.append((row['reviewText'], 1))
            elif row['scoreSentiment'] == "NEGATIVE":
                data.append((row['reviewText'], 0))


        train_size = int(0.8 * len(data))
        test_size = len(data) - train_size

        trainData = data[:train_size]
        textToTrainTO = []
        for text, label in trainData:
            textToTrainTO.append(text)

        for i in range(len(textToTrainTO)):
            if i > 10:
                break

        tokenizer.train_from_iterator(textToTrainTO, trainer=tokenizer_trainer)
        self.tokenizer = tokenizer

        self.data = data[:train_size] if train else data[train_size:]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        encoding = self.tokenizer.encode(text)

        return torch.tensor(encoding.ids, dtype=torch.float), label



In [10]:
my_dataset = SentimentDataset('rotten_tomatoes_movie_reviews.csv', train=True)
my_loader = data.DataLoader(my_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

ParserError: Error tokenizing data. C error: Expected 11 fields in line 973232, saw 13


In [10]:
encodings, labels = next(iter(my_loader))

for encoding, label in zip(encodings, labels):
    print(f"Encoding: {encoding}")
    print(f"text: {my_dataset.tokenizer.decode(encoding.tolist())}")
    print(f"Labels: {label}")

Encoding: tensor([2408,  270, 2740,  207, 8860,  542,   12, 5856,  197,   65, 3055,  197,
        1523, 4689, 1581,  324, 1359,   78, 2586, 1170,  406,  213,  181,   12,
         186, 1852,  181, 6470,  488, 2802,  187,  298, 1413, 1825,    7,   83,
        7500, 4530,   14,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])
text: Em man uel le Dev os , veteran of a number of relationship dramas directed by Ar n aud Des ple ch in , is terrific in subtly imp ressing on us Car la ' s outsider status .                         
Labels: 1
Encoding: tensor([  40,  922,  182,  706,  204, 2951, 1935,  337,  349,   65, 9036, 3671,
         197,  188, 5141, 1034,  327,   65, 1110,  518,  219, 1455,  707,    7,
          84,  217, 8325,  487,   14, 1539,  678,  683,  181, 9224,  383,  269,
         858,  219,  345,   31, 4228,  188,  513,  272,   14,    0,  

__Classification algorithm__

In [8]:
input_features = sequence_length
output_features = 2

output_Features1 = 32
output_Features2 = 64

output_Features_final = 64

number_of_conv_layers = 2
number_of_linear_layers = 2

for i in range(number_of_conv_layers):
    input_features = (input_features - 2) // 2


class CNN(nn.Sequential):
    def __init__(self):
        super().__init__(
            nn.Conv1d(1, 32, 3),
            nn.ReLU(),
            nn.MaxPool1d(2, stride=2),
            nn.Conv1d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool1d(2, stride=2),
            nn.Flatten(),
            nn.Linear(64 * output_Features_final, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_features),
        )

network = CNN()

device = torch.device(device)
network.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(network.parameters(), lr=learning_rate)

my_trainset = SentimentDataset('rotten_tomatoes_movie_reviews.csv', train=True)
my_trainloader = data.DataLoader(my_trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

my_testset = SentimentDataset('rotten_tomatoes_movie_reviews.csv', train=False)
my_testloader = data.DataLoader(my_testset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

def train(num_epochs):
    train_avg_loss = []
    test_avg_loss = []
    test_accuracy = []

    for i in range(num_epochs):
        print(f"Epoch number : {i+1}")
        train_losses = []
        test_losses = []

        for x, y in my_trainloader:
            x, y = x.to(device), y.to(device)

            pred = network(x)
            loss = criterion(pred, y)
            train_losses.append(loss.detach())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            for x, y in my_testloader:
                x, y = x.to(device), y.to(device)

                pred = network(x)
                loss = criterion(pred, y)
                test_losses.append(loss.detach())

                y_pred = pred.argmax(dim=1)
                correct = correct + (y_pred == y).sum()

            accuracy = correct / len(my_testset)

        test_accuracy.append(accuracy.cpu())
        train_avg_loss.append(torch.mean(torch.tensor(train_losses)))
        test_avg_loss.append(torch.mean(torch.tensor(test_losses)))

    return train_avg_loss, test_avg_loss, test_accuracy


ParserError: Error tokenizing data. C error: Expected 11 fields in line 973232, saw 13


In [18]:
train_avg_loss, test_avg_loss, test_accuracy = train(num_epochs)

Epoch number : 1


RuntimeError: Given groups=1, weight of size [32, 1, 3], expected input[1, 4, 64] to have 1 channels, but got 4 channels instead

In [ ]:
fig, axs = plt.subplots(3, 1)

axs[0].set_title('Training Average Loss')
axs[0].plot(train_avg_loss)
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss ')

axs[1].set_title('Testing Average Loss')
axs[1].plot(test_avg_loss)
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Loss ')

axs[2].set_title('Testing Accuracy')
axs[2].plot(test_accuracy)
axs[2].set_xlabel('Epochs')
axs[2].set_ylabel('Accuracy')

plt.tight_layout()
plt.show()